# Certified polynomial one-jet reduction benchmarks

This notebook compares interval Jacobians, unreduced polynomial Jacobians, and three sound support-reduction policies. The 100-dimensional target is the saved Poisson PINN checkpoint; running the benchmark never retrains it. Reduced terms are not discarded: their componentwise remainder is propagated rigorously and attached as fresh pointwise residual symbols at the output. Both squared integrations return scalar polynomial zonotopes; intervalization is performed only afterward to obtain the final norm bounds. Tightness is assessed by the actual intervals and the absolute and relative widths of both the final certified $L^2$ and $W^{1,2}$ norm intervals. Before integration, we also report the mean, maximum, and relative mean componentwise widths of the full Jacobian enclosure.

In [1]:
from collections import Counter
from math import sqrt
from pathlib import Path
from time import perf_counter
import pandas as pd
import torch
from intervalnets import (IntervalTensor, PZIntegrationCell, enable_interval_eval,
    integrate_pz_onejet_squared, integrate_pz_value_squared,
    load_tanh_mlp_checkpoint)
torch.set_num_threads(1)
torch.set_default_dtype(torch.float64)
enable_interval_eval()
repo_root = Path.cwd()
while not (repo_root / 'notebooks' / 'checkpoints').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
CHECKPOINT = repo_root / 'notebooks' / 'checkpoints' / 'pinn_100d_poisson.pt'

In [2]:
def make_model(input_dim, hidden=(50, 50, 50), seed=20260731):
    torch.manual_seed(seed)
    layers, previous = [], input_dim
    for width in hidden:
        layers += [torch.nn.Linear(previous, width), torch.nn.Tanh()]
        previous = width
    layers.append(torch.nn.Linear(previous, 1))
    return torch.nn.Sequential(*layers)

def norm_interval(squared):
    return (sqrt(max(0.0, float(squared.lower))), sqrt(max(0.0, float(squared.upper))))

def interval_metrics(bounds, prefix):
    lower, upper = map(float, bounds)
    absolute_width = upper - lower
    scale = max(abs(lower), abs(upper))
    relative_width = absolute_width / scale if scale > 0.0 else 0.0
    return {
        f'{prefix}_lower': lower,
        f'{prefix}_upper': upper,
        f'{prefix}_absolute_width': absolute_width,
        f'{prefix}_relative_width': relative_width,
    }

def jacobian_width_metrics(enclosure):
    lower = torch.as_tensor(enclosure.lower)
    upper = torch.as_tensor(enclosure.upper)
    widths = upper - lower
    scales = torch.maximum(lower.abs(), upper.abs())
    relative_widths = torch.where(scales > 0.0, widths / scales, 0.0)
    return {
        'J_mean_component_width_before_integration': float(widths.mean()),
        'J_max_component_width_before_integration': float(widths.max()),
        'J_relative_mean_component_width_before_integration': float(relative_widths.mean()),
    }

def benchmark(model, box, strategy='topk', **kwargs):
    cell = PZIntegrationCell.from_affine_box(box)
    start = perf_counter()
    traced = model.eval_pz_onejet(cell.domain, return_trace=True,
        reduction_strategy=strategy, **kwargs)
    forward_s = perf_counter() - start
    enclosure = traced.final.J.interval_enclosure()
    jacobian_metrics = jacobian_width_metrics(enclosure)
    start = perf_counter()
    l2_integrated_pz = integrate_pz_value_squared(traced.final.Y, cell, output='pz')
    l2_squared = l2_integrated_pz.interval_enclosure()
    l2_integration_s = perf_counter() - start
    start = perf_counter()
    w12_integrated_pz = integrate_pz_onejet_squared(traced.final, cell, output='pz')
    w12_squared = w12_integrated_pz.interval_enclosure()
    w12_integration_s = perf_counter() - start
    return {
        'strategy': strategy, **kwargs, 'forward_s': forward_s,
        'L2_integration_s': l2_integration_s,
        'W12_integration_s': w12_integration_s,
        'total_s': forward_s + w12_integration_s,
        'J_terms': len(traced.final.J.terms),
        'J_degree': max(map(sum, traced.final.J.terms), default=0),
        'noise': traced.final.J.num_noise,
        'L2_integrated_PZ_terms': len(l2_integrated_pz.terms),
        'W12_integrated_PZ_terms': len(w12_integrated_pz.terms),
        'W12_integrated_PZ_noise': w12_integrated_pz.num_noise,
        **jacobian_metrics,
        **interval_metrics(norm_interval(l2_squared), 'L2'),
        **interval_metrics(norm_interval(w12_squared), 'W12'),
        'trace': traced.records,
    }

## Small-network exact reference
The unreduced path is practical here and provides the polynomial reference endpoint.

In [3]:
small_model = make_model(8, hidden=(10, 10))
small_box = IntervalTensor.from_bounds([-0.15] * 8, [0.15] * 8)
small_exact = benchmark(small_model, small_box, strategy='none', reduce=False)
small_reduced = [benchmark(small_model, small_box, strategy=s, max_terms=24,
    max_degree=2, pca_rank=3, pca_candidates=24) for s in ('topk', 'degree', 'pca')]
small_reference_table = pd.DataFrame([
    {k: v for k, v in row.items() if k != 'trace'}
    for row in [small_exact, *small_reduced]
])
small_reference_table

,strategy,reduce,forward_s,L2_integration_s,W12_integration_s,total_s,J_terms,J_degree,noise,L2_integrated_PZ_terms,W12_integrated_PZ_terms,W12_integrated_PZ_noise,J_mean_component_width_before_integration,J_max_component_width_before_integration,J_relative_mean_component_width_before_integration,L2_lower,L2_upper,L2_absolute_width,L2_relative_width,W12_lower,W12_upper,W12_absolute_width,W12_relative_width,max_terms,max_degree,pca_rank,pca_candidates
0,none,False,0.014716,0.001302,0.029050,0.043766,142,2,36,1,1,29,0.023829,0.033046,0.390604,0.002189,0.002266,0.000077,0.033895,0.002536,0.002775,0.000239,0.086188,NaN,NaN,NaN,NaN
1,topk,NaN,0.009499,0.000707,0.003600,0.013099,24,2,36,1,1,29,0.024067,0.033327,0.393804,0.002189,0.002266,0.000077,0.033895,0.002532,0.002779,0.000247,0.088994,24.0,2.0,3.0,24.0
2,degree,NaN,0.009076,0.000671,0.003609,0.012685,24,2,36,1,1,29,0.024067,0.033327,0.393804,0.002189,0.002266,0.000077,0.033895,0.002532,0.002779,0.000247,0.088994,24.0,2.0,3.0,24.0
3,pca,NaN,0.010617,0.000707,0.003979,0.014596,27,2,39,1,1,32,0.024246,0.033514,0.396248,0.002189,0.002266,0.000077,0.033895,0.002531,0.002780,0.000249,0.089397,24.0,2.0,3.0,24.0


## Saved 100D Poisson PINN: 100–50–50–50–1
The trained candidate is loaded from the committed checkpoint; there is no optimizer or training loop in this notebook. All policies below therefore certify exactly the same saved PINN weights. They preserve a Jacobian polynomial core. The interval result is the speed/looseness baseline. The primary comparison quantities are the absolute and relative widths of both norm intervals; the Jacobian width diagnostics measure tightness before the integration step.

In [4]:
model = load_tanh_mlp_checkpoint(CHECKPOINT)
assert sum(parameter.numel() for parameter in model.parameters()) == 10201
display(pd.DataFrame([{
    'loaded_checkpoint': str(CHECKPOINT.relative_to(repo_root)),
    'training_steps': 0,
}]))
box = IntervalTensor.from_bounds([-0.1] * 100, [0.1] * 100)
configs = [
    ('topk-32', 'topk', dict(max_terms=32)),
    ('topk-64', 'topk', dict(max_terms=64)),
    ('topk-96', 'topk', dict(max_terms=96)),
    ('topk-128', 'topk', dict(max_terms=128)),
    ('topk-192', 'topk', dict(max_terms=192)),
    ('topk-256', 'topk', dict(max_terms=256)),
    ('degree-64', 'degree', dict(max_terms=64, max_degree=2)),
    ('pca-64', 'pca', dict(max_terms=64, pca_rank=4, pca_candidates=32)),
]
rows = []
for label, strategy, kwargs in configs:
    row = benchmark(model, box, strategy=strategy, **kwargs)
    row['label'] = label
    rows.append(row)
start = perf_counter()
interval_bound = model.sobolev_norm(box, p=2.0, order=1, method='interval')
interval_s = perf_counter() - start
interval_l2_bound = model.lpnorm(box, p=2.0, method='interval')
interval_jacobian = model.eval_jacobian(box)
interval_row = {
    'label': 'interval',
    'total_s': interval_s,
    **jacobian_width_metrics(interval_jacobian),
    **interval_metrics((interval_l2_bound.lower, interval_l2_bound.upper), 'L2'),
    **interval_metrics((interval_bound.lower, interval_bound.upper), 'W12'),
}
summary = [{k: v for k, v in row.items() if k != 'trace'} for row in rows]
benchmark_table = pd.DataFrame([interval_row, *summary]).set_index('label')
benchmark_table

,loaded_checkpoint,training_steps
0,notebooks/checkpoints/pinn_100d_poisson.pt,0


,total_s,J_mean_component_width_before_integration,J_max_component_width_before_integration,J_relative_mean_component_width_before_integration,L2_lower,L2_upper,L2_absolute_width,L2_relative_width,W12_lower,W12_upper,W12_absolute_width,W12_relative_width,strategy,max_terms,forward_s,L2_integration_s,W12_integration_s,J_terms,J_degree,noise,L2_integrated_PZ_terms,W12_integrated_PZ_terms,W12_integrated_PZ_noise,max_degree,pca_rank,pca_candidates
label,,,,,,,,,,,,,,,,,,,,,,,,,,
interval,0.194282,17.548815,20.864813,1.989760,0.0,8.984144e-35,8.984144e-35,1.0,0.0,1.000326e-33,1.000326e-33,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
topk-32,1.109944,17.112410,20.290220,1.981437,0.0,3.013843e-35,3.013843e-35,1.0,0.0,9.760069e-34,9.760069e-34,1.0,topk,32.0,0.527356,0.020018,0.582588,132.0,1.0,350.0,1.0,1.0,251.0,NaN,NaN,NaN
topk-64,1.600538,16.976826,20.127467,1.981290,0.0,3.013843e-35,3.013843e-35,1.0,0.0,9.683504e-34,9.683504e-34,1.0,topk,64.0,0.838902,0.012539,0.761636,164.0,1.0,350.0,1.0,1.0,251.0,NaN,NaN,NaN
topk-96,1.984816,16.849883,19.977130,1.981151,0.0,3.013843e-35,3.013843e-35,1.0,0.0,9.611825e-34,9.611825e-34,1.0,topk,96.0,1.157304,0.012495,0.827512,196.0,1.0,350.0,1.0,1.0,251.0,NaN,NaN,NaN
topk-128,2.373610,16.821834,19.945027,1.981119,0.0,3.013843e-35,3.013843e-35,1.0,0.0,9.595997e-34,9.595997e-34,1.0,topk,128.0,1.364302,0.012646,1.009308,226.0,1.0,350.0,1.0,1.0,251.0,NaN,NaN,NaN
topk-192,2.799860,16.789867,19.907969,1.981084,0.0,3.013843e-35,3.013843e-35,1.0,0.0,9.577961e-34,9.577961e-34,1.0,topk,192.0,1.553555,0.013095,1.246304,278.0,1.0,350.0,1.0,1.0,251.0,NaN,NaN,NaN
topk-256,2.964446,16.765920,19.879198,1.981057,0.0,3.013843e-35,3.013843e-35,1.0,0.0,9.564447e-34,9.564447e-34,1.0,topk,256.0,1.698161,0.013227,1.266284,300.0,1.0,350.0,1.0,1.0,251.0,NaN,NaN,NaN
degree-64,1.565116,16.976826,20.127467,1.981290,0.0,3.013843e-35,3.013843e-35,1.0,0.0,9.683504e-34,9.683504e-34,1.0,degree,64.0,0.873105,0.012746,0.692011,164.0,1.0,350.0,1.0,1.0,251.0,2.0,NaN,NaN
pca-64,1.609048,16.955616,20.104079,1.981267,0.0,3.013843e-35,3.013843e-35,1.0,0.0,9.671521e-34,9.671521e-34,1.0,pca,64.0,0.939264,0.012786,0.669784,168.0,1.0,362.0,1.0,1.0,263.0,NaN,4.0,32.0


In [5]:
assert all(row['total_s'] < 3.0 for row in rows), summary
start = perf_counter()
public_l2_bound = model.pz_l2norm(box)
public_l2_s = perf_counter() - start
start = perf_counter()
public_w12_bound = model.pz_sobolev_norm(box, order=1)
public_w12_s = perf_counter() - start
public_default_table = pd.DataFrame([{
    'public_default_L2_s': public_l2_s,
    'public_default_W12_s': public_w12_s,
    **interval_metrics((public_l2_bound.lower, public_l2_bound.upper), 'public_default_L2'),
    **interval_metrics((public_w12_bound.lower, public_w12_bound.upper), 'public_default_W12'),
}])
public_default_table

,public_default_L2_s,public_default_W12_s,public_default_L2_lower,public_default_L2_upper,public_default_L2_absolute_width,public_default_L2_relative_width,public_default_W12_lower,public_default_W12_upper,public_default_W12_absolute_width,public_default_W12_relative_width
0,0.064574,2.069711,-4.940656e-324,3.013843e-35,3.013843e-35,1.0,-4.940656e-324,9.611825e-34,9.611825e-34,1.0


## Layer diagnostics
The activation rows expose where support generation and certified tail growth occur. The benchmark-level Jacobian widths above are computed after the complete one-jet has been constructed but before either squared integral is evaluated. They use the full enclosure $J_{\mathrm{core}}+[-R,R]$, not merely the reduction remainder. For each entry, the relative width is $(\overline J_{ij}-\underline J_{ij})/\max(|\underline J_{ij}|,|\overline J_{ij}|)$, with exact-zero entries assigned zero; the reported relative mean is the mean of these componentwise ratios.

In [6]:
chosen = next(row for row in rows if row['label'] == 'topk-96')
layer_diagnostics = pd.DataFrame([{
    'module_index': record.layer_index,
    'layer': record.layer_type,
    'seconds': record.elapsed_s,
    'Y_terms': record.summary['Y']['term_count'],
    'J_terms': record.summary['J']['term_count'],
    'J_degree': record.summary['J']['max_degree'],
    'remainder_mean_radius': record.summary['J']['remainder_mean_radius'],
    'remainder_max_radius': record.summary['J']['remainder_max_radius'],
} for record in chosen['trace']])
layer_diagnostics

,module_index,layer,seconds,Y_terms,J_terms,J_degree,remainder_mean_radius,remainder_max_radius
0,-1,Input,0.000000,100,0,0,0.000000,0.000000
1,0,Linear,0.002594,100,0,0,0.000000,0.000000
2,1,Tanh,0.024830,150,96,1,0.026447,0.075139
3,2,Linear,0.008919,150,96,1,0.152948,0.232838
4,3,Tanh,0.477139,200,86,1,0.173841,0.279970
5,4,Linear,0.010834,200,86,1,1.039035,1.504241
6,5,Tanh,0.594485,250,96,1,1.067999,1.518824
7,6,Linear,0.011649,250,96,1,8.423916,9.987156


## Initial activation-approximation errors by neuron

For hidden layer $\ell$ and neuron $i$, the incoming value PZ is intervalized once to obtain the preactivation interval $I_{\ell i}$. That same interval is used for both certified affine enclosures

$$\tanh(z)\in p_{\ell i}^{(0)}z+q_{\ell i}^{(0)}+\delta_{\ell i}^{(0)}[-1,1],$$

$$\tanh'(z)\in p_{\ell i}^{(1)}z+q_{\ell i}^{(1)}+\delta_{\ell i}^{(1)}[-1,1].$$

The tables below show the actual certified approximation-noise coefficients $\delta_{\ell i}^{(0)}$ and $\delta_{\ell i}^{(1)}$ before multiplication by the incoming Jacobian and before Top-$k$/PCA reduction. Rows are one-based hidden-neuron indices; columns are activation layers.

In [7]:
activation_records = [record for record in chosen['trace'] if record.layer_type == 'Tanh']
assert len(activation_records) == 3
assert all(record.summary['tanh_approximation_radii'].numel() == 50 for record in activation_records)

def per_neuron_activation_error_table(summary_key):
    columns = {
        f'hidden_layer_{layer_index}': record.summary[summary_key].detach().cpu().numpy()
        for layer_index, record in enumerate(activation_records, start=1)
    }
    frame = pd.DataFrame(columns)
    frame.index = pd.RangeIndex(1, len(frame) + 1, name='neuron')
    return frame

tanh_value_approximation_errors = per_neuron_activation_error_table(
    'tanh_approximation_radii'
)
tanh_value_approximation_errors

,hidden_layer_1,hidden_layer_2,hidden_layer_3
neuron,,,
1,0.089748,0.080818,0.129496
2,0.062994,0.080196,0.147882
3,0.087721,0.092721,0.094163
4,0.068898,0.073458,0.143326
5,0.076235,0.081105,0.123203
6,0.071089,0.069578,0.129580
7,0.092842,0.082595,0.107198
8,0.078379,0.063086,0.121686
9,0.079325,0.097897,0.160625


### Certified $\tanh'$ approximation errors

In [8]:
tanh_prime_approximation_errors = per_neuron_activation_error_table(
    'tanh_prime_approximation_radii'
)
tanh_prime_approximation_errors

,hidden_layer_1,hidden_layer_2,hidden_layer_3
neuron,,,
1,0.303512,0.288282,0.357791
2,0.253699,0.287270,0.378499
3,0.300122,0.307124,0.310248
4,0.265936,0.274893,0.373685
5,0.279597,0.288077,0.350153
6,0.270216,0.266842,0.357862
7,0.308496,0.291039,0.328011
8,0.283930,0.254079,0.348642
9,0.285594,0.316354,0.390683


### Per-layer approximation-error summary

In [9]:
activation_error_summary = pd.DataFrame([{
    'hidden_layer': layer_index,
    'tanh_delta_min': record.summary['tanh_approximation_radius_min'],
    'tanh_delta_mean': record.summary['tanh_approximation_radius_mean'],
    'tanh_delta_max': record.summary['tanh_approximation_radius_max'],
    'tanh_prime_delta_min': record.summary['tanh_prime_approximation_radius_min'],
    'tanh_prime_delta_mean': record.summary['tanh_prime_approximation_radius_mean'],
    'tanh_prime_delta_max': record.summary['tanh_prime_approximation_radius_max'],
} for layer_index, record in enumerate(activation_records, start=1)]).set_index('hidden_layer')
activation_error_summary

,tanh_delta_min,tanh_delta_mean,tanh_delta_max,tanh_prime_delta_min,tanh_prime_delta_mean,tanh_prime_delta_max
hidden_layer,,,,,,
1,0.052128,0.074179,0.101042,0.229436,0.275129,0.321017
2,0.063086,0.092422,0.128366,0.254079,0.305928,0.356997
3,0.067150,0.133023,0.202250,0.261997,0.359351,0.423880


## Interpretation

- Top-k is the cheapest reduction and gives a direct runtime/tightness knob.
- Both squared integrations return scalar PZs. Pointwise uncertainty becomes a fresh integrated global generator; intervalization occurs only after this integration step.
- For both $L^2$ and $W^{1,2}$, the primary final tightness diagnostic is $U-L$ for the certified norm interval $[L,U]$. The reported relative width is $(U-L)/\max(|L|,|U|)$ (and is defined as zero when both endpoints vanish); it is not obtained by dividing by the lower bound.
- The mean, maximum, and relative mean Jacobian component widths are complementary pre-integration diagnostics: they show how much tightness has already been lost in the image enclosure, before squaring and integration can add further overestimation. The relative mean averages the entrywise width divided by the largest endpoint magnitude, so it is scale-normalized and lies between zero and two.
- In the target experiment every method currently has $L=0$ for both norms, hence every relative norm width is $100\%$. Here a smaller upper endpoint happens to equal a smaller absolute width, but it does not constitute an improvement in relative precision.
- The target-network relative mean Jacobian widths are close to two. This says that most component intervals straddle zero and are nearly symmetric relative to their endpoint magnitude. The absolute mean and maximum widths therefore remain the more discriminating Jacobian diagnostics in this experiment.
- The full per-neuron tables separate the initial activation-value error from the derivative error. Across hidden layers 1--3, the mean $\tanh$ radii are approximately $0.0742$, $0.0924$, and $0.1330$, whereas the mean $\tanh'$ radii are approximately $0.2751$, $0.3059$, and $0.3594$. Thus the derivative enclosure is already the larger local error source before Jacobian multiplication and support reduction.
- Degree capping matters once higher-degree terms survive the importance ranking; on narrow boxes it can coincide with top-k.
- PCA is certified because the projected generators are intervalized in PCA coordinates and the orthogonal residual is bounded componentwise. Its SVD and added pointwise generators must earn their cost empirically.
- Extending the target sweep from Top-96 through Top-256 shows clear saturation: Top-256 remains just below three seconds in this run but improves the $W^{1,2}$ width by only about $0.49\%$ relative to Top-96. Top-192 is the more robust sub-three-second accuracy-biased configuration.
- The unreduced polynomial path is intentionally limited to smaller networks: it diagnoses genuine monomial growth rather than hiding it behind interval propagation.